# CURE-Rec — remaining review closure

This notebook is the single control surface for the remaining review items. It never edits the registered YAML configuration. Run all cells from top to bottom. Actions that require unavailable data or a new policy implementation are recorded as explicit skips; no result is fabricated.


In [ ]:
from pathlib import Path
import sys,json,re,time
import pandas as pd

C=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in C if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_suite import objective_ablation, paired_user_statistics

CONFIG=ROOT/'configs'/'curesim_full.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'
PAPER=ROOT.parent/'paper'/'cure-rec-springer'/'cure-rec.tex'
RUN_ALL=True
RUN_DIVERGENT_SELECTOR=True
RUN_UTILITY_SENSITIVITY=True
RUN_CONSTRAINT_FRONTIER=True
RUN_SEMI_REAL=False  # requires a declared replay/world-model protocol
RUN_INTEGRATED_SCALING=False  # requires actual 8/10 policy operators
print('Root:',ROOT)

## 1. Manuscript consistency audit


In [ ]:
text=PAPER.read_text()
checks={
 'free_F4_user_index': bool(re.search(r'Q_\{j,t\+1\}.*1\{j\\in L_\{u,t\}\}',text)),
 'old_decision_faithful': 'decision-faithful attribution' in text,
 'old_preregistered': 'preregistered' in text,
 'old_reviewer_language': 'reviewer-revision checks' in text or 'Reviewer-revision evidence' in text,
 'table_3_reference': 'Table 3' in text,
 'artifact_section': 'Artifact inventory' in text,
 'cost_definition': 'c(S)=\\sum' in text,
 'gini_definition': 'G(x)=' in text,
}
print(json.dumps(checks,indent=2))
if checks['old_decision_faithful'] or checks['old_preregistered'] or checks['old_reviewer_language'] or checks['free_F4_user_index']:
    print('Manuscript still has terminology/notation items requiring manual editing.')

## 2. Utility-weight sensitivity

Runs predeclared alternative utility vectors on one exact full game each. Results are sensitivity evidence, not performance gains.

In [ ]:
WEIGHTS=[
 {'name':'satisfaction_heavy','satisfaction_weight':0.85,'retention_weight':0.15,'fatigue_weight':0.35,'cost_weight':1.0},
 {'name':'retention_heavy','satisfaction_weight':0.35,'retention_weight':0.65,'fatigue_weight':0.35,'cost_weight':1.0},
 {'name':'fatigue_averse','satisfaction_weight':0.70,'retention_weight':0.30,'fatigue_weight':0.70,'cost_weight':1.0},
 {'name':'cost_averse','satisfaction_weight':0.70,'retention_weight':0.30,'fatigue_weight':0.35,'cost_weight':2.0},
]
if RUN_ALL and RUN_UTILITY_SENSITIVITY:
 rows=[]
 for w in WEIGHTS:
  cfg=load_settings(CONFIG); cfg.run.name='utility-'+w['name']; cfg.run.output_root=ROOT/'runs'/'reviewer-closure';
  for k,v in w.items():
   if k!='name': setattr(cfg.utility,k,v)
  logger,game,decision=run_experiment(cfg)
  rows.append({'setting':w['name'],'portfolio':';'.join(decision.selected_interventions),'status':decision.status.value,'mode':decision.mode.value,'lower_improvement':decision.lower_improvement,'base_feasible':decision.base_feasible,'source_run':str(logger.run_dir)})
 utility=pd.DataFrame(rows); out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True); utility.to_csv(out/'utility_weight_sensitivity.csv',index=False); display(utility)
else: print('Utility sensitivity skipped.')

## 3. Constraint frontier

Runs a predeclared grid of budget, relevance, provider and fatigue thresholds.


In [ ]:
FRONTIER=[
 {'name':'strict_provider','budget':0.35,'relevance':-0.08,'provider':0.24,'fatigue':0.65},
 {'name':'baseline','budget':0.35,'relevance':-0.08,'provider':0.28,'fatigue':0.65},
 {'name':'relaxed_provider','budget':0.35,'relevance':-0.08,'provider':0.34,'fatigue':0.65},
 {'name':'tight_budget','budget':0.20,'relevance':-0.08,'provider':0.28,'fatigue':0.65},
 {'name':'tight_relevance','budget':0.35,'relevance':-0.03,'provider':0.28,'fatigue':0.65},
 {'name':'tight_fatigue','budget':0.35,'relevance':-0.08,'provider':0.28,'fatigue':0.45},
]
if RUN_ALL and RUN_CONSTRAINT_FRONTIER:
 rows=[]
 for f in FRONTIER:
  cfg=load_settings(CONFIG); cfg.run.name='frontier-'+f['name']; cfg.run.output_root=ROOT/'runs'/'reviewer-closure'; cfg.constraints.budget=f['budget']; cfg.constraints.min_relevance_delta=f['relevance']; cfg.constraints.max_provider_disparity=f['provider']; cfg.constraints.max_fatigue=f['fatigue']
  logger,game,decision=run_experiment(cfg); rows.append({'setting':f['name'],**f,'portfolio':';'.join(decision.selected_interventions),'status':decision.status.value,'mode':decision.mode.value,'lower_improvement':decision.lower_improvement,'base_feasible':decision.base_feasible,'source_run':str(logger.run_dir)})
 frontier=pd.DataFrame(rows); out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True); frontier.to_csv(out/'constraint_frontier.csv',index=False); display(frontier)
else: print('Constraint frontier skipped.')

## 4. Divergent selector protocol

The primary full configuration has selector ties. This cell records the mandatory protocol: select only on calibration seeds, freeze masks, and evaluate on disjoint seeds. A full implementation must use completed game tables from configurations where masks differ; it must not treat a single dominant repeat-cap configuration as evidence of selector superiority.

In [ ]:
if RUN_ALL and RUN_DIVERGENT_SELECTOR:
 print('Divergent-selector action requires archived configurations with different masks; primary tied result is retained as a null comparison.')
 print('Required output columns: configuration, selector, selection_seed, evaluation_seed, frozen_mask, robust_regret, feasible, constraint margins.')
else: print('Divergent selector protocol skipped.')

## 5. Semi-real intervention validation


In [ ]:
if RUN_SEMI_REAL:
 raise NotImplementedError('Requires audited replay/world-model intervention semantics; no unsupported real intervention claim is generated.')
else: print('Semi-real intervention validation skipped: audited intervention log/world model unavailable.')

## 6. Integrated 8/10-player scaling


In [ ]:
if RUN_INTEGRATED_SCALING:
 raise NotImplementedError('Distinct 8/10-player policy operators must be integrated into CURE-Sim before this study can run.')
else: print('Integrated scaling skipped: current 8/10 result is arithmetic attribution stress testing only.')

In [ ]:
out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True)
manifest={'run_all':RUN_ALL,'utility_sensitivity':'executed' if RUN_UTILITY_SENSITIVITY else 'skipped','constraint_frontier':'executed' if RUN_CONSTRAINT_FRONTIER else 'skipped','divergent_selector':'protocol_only_primary_tie_retained','semi_real':'skipped_no_audited_intervention_data','integrated_scaling':'skipped_no_integrated_8_10_operators','yaml_changed':False}
(out/'closure_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2))